# 04 — Keyword Extraction
This notebook compares two keyword extraction approaches used in ClauseGuard:

| Method | Best for | How it works |
|---|---|---|
| **TF-IDF** | Multiple clauses | Finds words that are important in one clause *relative to all others* |
| **YAKE** | A single clause | Statistical heuristics — position, frequency, co-occurrence |

**Production file:** `backend/keywords.py`

In [ ]:
import sys
sys.path.append('..')

from sklearn.feature_extraction.text import TfidfVectorizer
import yake
import numpy as np

## 1. Load and segment the contract

In [ ]:
from backend.extractor import extract_text
from backend.cleaner import clean_text
from backend.segmenter import segment_into_clauses

raw = extract_text("../tests/sample_contract.pdf")
cleaned = clean_text(raw)
clauses = segment_into_clauses(cleaned)

print(f"Contract segmented into {len(clauses)} clauses")
for i, c in enumerate(clauses[:3], 1):
    print(f"\nClause {i}: {c[:100]}...")

## 2. TF-IDF Keyword Extraction (per clause, across all clauses)

In [ ]:
def extract_keywords_tfidf(clauses, top_n=5):
    """
    TF-IDF scores each word by how important it is in a clause
    RELATIVE to all other clauses.
    High TF-IDF = rare across the document but frequent in this clause.
    """
    if len(clauses) < 2:
        return [[] for _ in clauses]

    vectorizer = TfidfVectorizer(stop_words="english", max_features=200)
    tfidf_matrix = vectorizer.fit_transform(clauses)
    feature_names = vectorizer.get_feature_names_out()

    results = []
    for row in tfidf_matrix:
        row_data = row.toarray()[0]
        top_indices = row_data.argsort()[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_indices if row_data[i] > 0]
        results.append(top_words)
    return results

tfidf_results = extract_keywords_tfidf(clauses)
print("TF-IDF Keywords per clause:")
for i, keywords in enumerate(tfidf_results, 1):
    print(f"  Clause {i}: {keywords}")

## 3. Understand TF-IDF scores with a heatmap

In [ ]:
import matplotlib.pyplot as plt

vectorizer = TfidfVectorizer(stop_words="english", max_features=20)
matrix = vectorizer.fit_transform(clauses).toarray()
terms = vectorizer.get_feature_names_out()

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(matrix, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(terms)))
ax.set_xticklabels(terms, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(clauses)))
ax.set_yticklabels([f"Clause {i+1}" for i in range(len(clauses))], fontsize=9)
ax.set_title("TF-IDF Score Heatmap — Clauses vs. Terms", fontsize=12)
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 4. YAKE Keyword Extraction (single clause)

In [ ]:
def extract_keywords_yake(text, top_n=5):
    """
    YAKE (Yet Another Keyword Extractor) works on a SINGLE document.
    It uses position, frequency, and word co-occurrence.
    Lower score = more important keyword.
    """
    kw_extractor = yake.KeywordExtractor(top=top_n, n=2)  # up to 2-word phrases
    keywords = kw_extractor.extract_keywords(text)
    return [(kw, round(score, 4)) for kw, score in keywords]

print("YAKE keywords for each clause (keyword, score — lower is more important):\n")
for i, clause in enumerate(clauses, 1):
    yake_kws = extract_keywords_yake(clause)
    print(f"Clause {i}: {yake_kws}")

## 5. Comparison: TF-IDF vs YAKE on the IP clause

In [ ]:
# Find the IP clause
ip_clause = next((c for c in clauses if 'intellectual property' in c.lower()), clauses[0])
print("Clause text:")
print(ip_clause)
print()

# TF-IDF (needs all clauses as context)
idx = clauses.index(ip_clause)
tfidf_kws = tfidf_results[idx]

# YAKE (standalone)
yake_kws = [kw for kw, _ in extract_keywords_yake(ip_clause)]

print(f"TF-IDF keywords : {tfidf_kws}")
print(f"YAKE keywords   : {yake_kws}")